# 01 · Your first Factor Weave request

> Goal: install, authenticate, pull one ticker's current factor row.

Time: 3 minutes.

## What we'll do

1. Set up a tiny client (works with or without an API key)
2. Pull AAPL's current factor row
3. Look at what's in it

In [1]:
# Setup — works with or without an API key.
# With FACTORWEAVE_API_KEY set, we use the full API (10,000+ tickers).
# Without one, we fall back to /demo/{ticker} (AAPL, MSFT, NVDA, AMZN, GOOGL, META, TSLA, JPM).
import os, json
import requests

API_BASE = "https://factorweave.com/api"
API_KEY = os.environ.get("FACTORWEAVE_API_KEY")
DEMO_TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "JPM"]
MODE = "live" if API_KEY else "demo"
print(f"Running in {MODE} mode.", "Key prefix:", (API_KEY[:8] + '…') if API_KEY else "(none)")


def fw_demo(ticker: str) -> dict:
    """Demo endpoint — no auth, 8 sample tickers, current snapshot only."""
    r = requests.get(f"{API_BASE}/demo/{ticker}", timeout=10)
    r.raise_for_status()
    return r.json()


def fw_features(ticker: str, **kwargs) -> dict:
    """Authed features endpoint when a key is available; demo fallback otherwise."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/features/{ticker}",
                         params=kwargs,
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — reshape demo response to look like the authed one
    d = fw_demo(ticker)
    return {"rows": [{"ticker": d["ticker"], "date": d["as_of"], **d["factors"]}]}


def fw_top(factor: str, n: int = 25) -> dict:
    """Top-N by a factor. Needs auth for the full universe; in demo mode we
    rank the 8 demo tickers locally."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/top",
                         params={"factor": factor, "n": n},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — fetch each demo ticker, sort locally
    rows = []
    for t in DEMO_TICKERS:
        d = fw_demo(t)
        if factor in d["factors"]:
            rows.append({"ticker": t, "date": d["as_of"], factor: d["factors"][factor]})
    rows.sort(key=lambda r: r[factor], reverse=True)
    return {"rows": rows[:n]}


def fw_similar(ticker: str, method: str = "cosine", limit: int = 10) -> dict:
    """Similarity search. Demo endpoint includes pre-computed `similar` set."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/vector-search/similar/{ticker}",
                         params={"method": method, "limit": limit, "min_lookback_days": 30},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — uses the `similar` array baked into the demo response
    d = fw_demo(ticker)
    return {"ticker": ticker, "method": "cosine (demo)", "neighbors": d.get("similar", [])[:limit]}


def fw_market_context() -> dict:
    """Universe analytics. Public on FREE, fuller on HOBBY+."""
    headers = {"X-API-Key": API_KEY} if API_KEY else {}
    r = requests.get(f"{API_BASE}/market-context", params={"latest": 1}, headers=headers, timeout=10)
    if r.status_code == 401:
        return {"_note": "market-context requires auth in demo mode"}
    r.raise_for_status()
    return r.json()


Running in demo mode. Key prefix: (none)


## Pull AAPL's current factor row

In [2]:
row = fw_features("AAPL")["rows"][0]
print("ticker:", row["ticker"])
print("as_of: ", row["date"])
print()
print("RSI:           ", round(row["rsi"], 2))
print("Momentum:      ", round(row["mom"], 4))
print("Realized vol:  ", round(row["rv_20"], 4))
print("Beta vs SPY:   ", round(row["beta_spy"], 3))
print("Composite z:   ", round(row["comp_score"], 4))
print("Composite rank:", row["q_comp_score"], "(percentile, 0-100)")

ticker: AAPL
as_of:  2026-05-22

RSI:            90.48
Momentum:       0.1393
Realized vol:   0.1853
Beta vs SPY:    0.909
Composite z:    0.1372
Composite rank: 55 (percentile, 0-100)


## What you just got

Each factor row is point-in-time for one trading day. The full row in the authed API has ~28 columns; the demo subset above is enough to feel the shape.

In live mode the same call returns history when you pass `start_date` and `end_date`, like:

```python
hist = fw_features("AAPL", start_date="2024-01-01", end_date="2024-12-31")
```

That's the 252-trading-day window for backtests.

## Next

→ `02-screening.ipynb` — rank the universe by a factor.